## Использование пакета lidR для сегментации LiDAR воздушного базирования

Официальный сайт lidR:
https://www.r-lidar.com/lidr

Подробное описание методов сегментации:
https://r-lidar.github.io/lidRbook/itd.html

Библиотека включает в себя 3 метода сегментации:

1.	Dalponte2016

Dalponte M., Coomes D.A. Tree‐centric mapping of forest carbon density from airborne laser scanning and hyperspectral data. // Methods in Ecology and Evolution. – 2016. - 7(10). - С. 1236-1245.

2.	Silva2016

Silva, C. A., Hudak, A. T., Vierling, L. A., Loudermilk, E. L., O’Brien, J. J., Hiers, J. K., Khosravipour, A.. Imputation of Individual Longleaf Pine (Pinus palustris Mill.) Tree Attributes from Field and LiDAR Data. // Canadian Journal of Remote Sensing – 2016. - №42(5) – С. 554–573.
  
3.	Li2012

W. Li, Q. Guo, M. K. Jakubowski, and M. Kelly, “A new method for segmenting individual trees from the Lidar point cloud,” // Photogramm. Eng. Remote Sens.- 2012. - №78(1). - С. 75–84.

Также возможно использование 4-го метода watershed, но его результаты неудовлетворительны.

In [ ]:
install.packages("lidR")  # установка библиотеки
install.packages("rgl")

## Набор данных

Набор данных LaPalma взят с https://zenodo.org/records/14051046

Включает в себя 25 деревьев, средняя плотность точек 520 точек на кв.м., место сканирования - Испании

Скачайте и распакуйте архив в директорию с данным ноутбуком

In [ ]:
options(warn=-1)  # Отключение предупреждений
library(lidR)
library(rgl)

options(rgl.printRglwidget = TRUE) # настройки отображения
options(rgl.useNULL = TRUE)
rgl::setupKnitr()

In [ ]:
# Пути для входного и выходного файлов
input_las  <- "./LIDAR_TreeSegmentation_LaPalma/Cloud_normalized_limoneros.las"
output_las <- "./LaPalma_lidR.las"

# Эталон сегментации
ref_las  <- readLAS("./LIDAR_TreeSegmentation_LaPalma/Cloud_segmented_limoneros.las")

# Импорт облака точек
las <- readLAS(input_las)
stopifnot(!is.empty(las))

cat("Загружено", npoints(las), "точек\n")

In [ ]:
# Вспомогательная функция для подсчета деревьев
count_trees <- function(tree_id) {
  ids <- unique(tree_id)
  ids <- ids[ids > 0]
  length(ids)
}

cat("Эталонное кол-во деревьев: ", count_trees(ref_las@data$Tree_ID_1))

Методы сегментации на основе растровых изображений предполагают преобразование исходного облака точек в растровое представление, такое как модель высот кроны CHM (Canopy Height Model). CHM представляет собой растровое изображение, где значение каждого пикселя соответствует максимальной высоте растительности над землей в соответствующей области. CHM строится для упрощения анализа трехмерных данных в двумерном пространстве, что позволяет применять методы обработки изображений.

Вершины деревьев выявляются как локальные максимумы в CHM, предполагая, что они соответствуют пикам высот крон. Используется фильтр локальных максимумов (Local Maxima Filter, LMF) с переменным размером окна для учета разных размеров деревьев. 

In [ ]:
# Разрешение для построения Canopy Height Model
chm_res <- 0.5

# Размер окна для поиска локальных максимумов
lmf_ws <- 7      

# Построение CMH и поиск вершин деревьев (локальных максимумов)
chm <- rasterize_canopy(las, res = chm_res, algorithm = p2r())
ttops <- locate_trees(chm, lmf(lmf_ws))

Далее задаются только основные параметры для алгоритмов сегментации. Полный список возможных параметров можно найти в документации:

https://cran.r-universe.dev/lidR/doc/manual.html#its_dalponte2016

### Dalponte2016

Метод Dalponte2016 основан на построении цифровой модели высот (CHM) и последующей сегментации крон деревьев с помощью анализа высотных пиков (локальных максимумов). Входными данными являются облако точек и CHM. Сначала определяются вершины деревьев, а затем вокруг каждой вершины проводится сегментация: точки, относящиеся к дереву, группируются на основе формы кроны и расстояния до найденного максимума. Использует адаптивный порог снижения высоты, вычисленный на основе разницы между пиком и текущим пикселем. Метод позволяет хорошо отделять кроны, но чувствителен к качеству CHM и плотности точек.

Параметры метода:
- Входные данные: трехмерное облако точек (las), растровая модель высот (CMH) и координаты вершин деревьев
- th_tree - порог минимальной высоты пикселя, чтобы считаться частью дерева (пиксели ниже заданной высоты считаются травой или шумом)
- th_seed - порог «роста» кроны относительно высоты дерева (пиксель добавляется, если он «достаточно высокий» относительно вершины)
- th_cr - порог «роста» относительно средней высоты текущего сегмента (пиксель добавляется, если «достаточно высокий» относительно среднего уровня текущей кроны)
- max_cr – максимальный диаметр кроны в пикселях


In [ ]:
# DALPONTE

cat("Начало сегментации Dalponte...\n")
t0 <- Sys.time()

las_d <- segment_trees(
  las,
  dalponte2016(chm = chm, treetops = ttops, th_tree = 2, th_seed=0.4, th_cr=0.5)
)

t1 <- Sys.time() # Засекаем время выполнения сегментации
time_d <- round(difftime(t1, t0, units = "secs"), 2)

dalponte_id <- las_d@data$treeID
dalponte_id[is.na(dalponte_id)] <- 0

cat("Сегментация завершена\n")
cat("  Время сегментации:  ", time_d, "с\n")
cat("  Кол-во деревьев: ", count_trees(dalponte_id), "\n\n")

### Silva2016

Алгоритм Silva2016 также использует цифровые модели высот CHM и локальные максимумы, но применяет метод растущих крон с использованием информации о расстоянии и высоте. Не использует фиксированный порог роста. Входные параметры включают CHM и список предполагаемых вершин деревьев. Метод устойчив к шуму и может работать при разных плотностях, но хуже справляется с перекрывающимися кронами.

Параметры метода:
- Входные данные: трехмерное облако точек (las), растровая модель высот (CMH) и координаты вершин деревьев
- exclusion - уровень отсеивания низких пикселей (доля высоты дерева), отсеивает нижние пиксели, не принадлежащие кроне
- max_cr – максимальный диаметр кроны в пикселях в доле от высоты дерева (динамически меняет допустимый диаметр от высоты дерева)


In [ ]:
# SILVA

cat("Начало сегментации Silva...\n")
t0 <- Sys.time()

las_s <- segment_trees(
  las,
  silva2016(chm = chm, treetops = ttops, max_cr = 6)
)

t1 <- Sys.time()
time_s <- round(difftime(t1, t0, units = "secs"), 2)

silva_id <- las_s@data$treeID
silva_id[is.na(silva_id)] <- 0

cat("Сегментация завершена\n")
cat("  Время сегментации:  ", time_s, "с\n")
cat("  Кол-во деревьев: ", count_trees(silva_id), "\n\n")

### Li2012
Метод Li2012 использует подход «top-down», начиная с поиска высоких точек, иерархически делит облако на сегменты, представляющие отдельные деревья. Он не требует CHM, работает напрямую с 3D-точками. Определяет локальные максимумы с использованием порогов и радиуса поиска. Использует разные пороги для высоких и низких точек. Основной плюс — независимость от растровых преобразований, но алгоритм чувствителен к плотности и равномерности распределения точек. 

Параметры метода:
- Входные данные: трехмерное облако точек (las)
- dt1 - порог для низких деревьев
- dt2 - порог для высоких деревьев
- R - радиус поиска локального максимума
- Zu - граница высоты, после которой используется dt2 вместо dt1
- hmin – минимальная высота для дерева

In [ ]:
# LI

cat("Начало сегментации Li...\n")
t0 <- Sys.time()

las_l <- segment_trees(
  las,
  li2012(R = 1.5, Zu = 10)
)

t1 <- Sys.time()
time_l <- round(difftime(t1, t0, units = "secs"), 2)

li_id <- las_l@data$treeID
li_id[is.na(li_id)] <- 0

cat("Сегментация завершена\n")
cat("  Время сегментации:  ", time_l, "с\n")
cat("  Кол-во деревьев: ", count_trees(li_id), "\n\n")

In [ ]:
cat("========================================================\n")
cat("Подведение итогов:\n")
cat(" Dalponte - ", count_trees(dalponte_id), " деревьев - ", time_d, " сек\n")
cat(" Silva    - ", count_trees(silva_id), " деревьев - ", time_s, " сек\n")
cat(" Li       - ", count_trees(li_id), " деревьев - ", time_l, " сек\n")
cat("========================================================\n")

In [ ]:
# Сохранение 3 аттрибутов в один LASiusx и экспорт резульатов

las <- add_lasattribute(
  las,
  dalponte_id,
  "Dalponte_ID",
  "uint32"
)

las <- add_lasattribute(
  las,
  silva_id,
  "Silva_ID",
  "uint32"
)

las <- add_lasattribute(
  las,
  li_id,
  "Li_ID",
  "uint32"
)

writeLAS(las, output_las)

In [ ]:
# Проверка, что аттрибуты были добавлены
las_check <- readLAS(output_las)
names(las_check@data)

## Визуализация

In [ ]:
plot(ref_las,
     color = "Tree_ID_1",
     size = 3,
     bg = "white")

rglwidget()

In [ ]:
plot(las,
     color = "Dalponte_ID",
     size = 3,
     bg = "white")

rglwidget()

In [ ]:
plot(las,
     color = "Silva_ID",
     size = 3,
     bg = "white")

rglwidget()

In [ ]:
plot(las,
     color = "Li_ID",
     size = 3,
     bg = "white")

rglwidget()